# Temperature

In [36]:
# Present Period with Standard Deviation Analysis - CORRECTED WEIGHTING

import xarray as xr
import numpy as np

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Seasonal extraction using xarray only ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_base_19712000_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_present_19812010_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_future126_20212050_seasonal.nc")

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d = weights.broadcast_like(base_reg["tas"].isel(year=0))

        # Calculate weighted spatial means for each year (in Kelvin)
        base_spatial_mean_K = (base_reg["tas"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        present_spatial_mean_K = (present_reg["tas"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        
        # Calculate period means in Kelvin
        base_mean_K = base_spatial_mean_K.mean(dim="year")
        present_mean_K = present_spatial_mean_K.mean(dim="year")
        
        # Calculate standard deviations in Kelvin
        base_std_K = base_spatial_mean_K.std(dim="year", ddof=1)
        present_std_K = present_spatial_mean_K.std(dim="year", ddof=1)
        
        # Convert means to Celsius for absolute differences
        base_mean_C = base_mean_K - 273.15
        present_mean_C = present_mean_K - 273.15
        
        # Absolute differences in Celsius
        diff_mean_C = present_mean_C - base_mean_C
        diff_std_C = present_std_K - base_std_K  # Std diff same in K or C
        
        # CORRECTED: Percentage changes based on Kelvin values
        pct_mean = ((present_mean_K - base_mean_K) / base_mean_K) * 100
        pct_std = ((present_std_K - base_std_K) / base_std_K) * 100

        results[region][season] = {
            "mean_diff_C": diff_mean_C.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_C": diff_std_C.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nTemperature Analysis (Present - Historical):")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(°C)':>12}{'Change':>10}{'(°C)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_C']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_C']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_C": metrics["mean_diff_C"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_C": metrics["std_diff_C"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_present_vs_baseline_with_std.csv", index=False)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_mid_SSP126_vs_baseline_with_std.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_C: Change in mean temperature (°C)")
print("- Mean_Pct_Change: Percentage change in mean temperature (% relative to Kelvin baseline)")
print("- Std_Diff_C: Change in standard deviation (°C)")
print("- Std_Pct_Change: Percentage change in standard deviation (% relative to Kelvin baseline)")


Temperature Analysis (Present - Historical):
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                    (°C)    Change        (°C)    Change
----------------------------------------------------------------------
Global              DJF            0.137      0.05       0.065     24.12
Global              MAM            0.129      0.04       0.066     27.29
Global              JJA            0.143      0.05       0.065     30.37
Global              SON            0.136      0.05       0.047     20.97
Tropics             DJF            0.066      0.02       0.068     16.50
Tropics             MAM            0.081      0.03       0.064     16.32
Tropics             JJA            0.075      0.03       0.043     11.24
Tropics             SON            0.062      0.02       0.042     10.40
Subtropics_N        DJF            0.258      0.09       0.050     17.85
Subtropics_N        MAM            0.275      0.09       0.052     19.67
Subtrop

# Precipitable Water

In [ ]:
# Present Period with Standard Deviation Analysis - CORRECTED WEIGHTING

import xarray as xr
import numpy as np

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Seasonal extraction using xarray only ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/prw/MIROC6_prw_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/prw/MIROC6_prw_future585_20212050_seasonal.nc")

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    #present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d = weights.broadcast_like(base_reg["prw"].isel(year=0))

        # CORRECTED: Calculate weighted mean for each period
        # Method 1: Calculate spatial weighted mean for each year, then temporal mean
        base_spatial_means = (base_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        present_spatial_means = (present_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        
        # Overall period means
        base_mean = base_spatial_means.mean(dim="year")
        present_mean = present_spatial_means.mean(dim="year")
        
        # CORRECTED: Calculate standard deviation from the spatially-averaged annual time series
        base_std = base_spatial_means.std(dim="year", ddof=1)
        present_std = present_spatial_means.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100
        
        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_mm": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mm": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nPrecipitable Water Analysis (Present - Historical):")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm)':>12}{'Change':>10}{'(mm)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mm']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mm']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mm": metrics["mean_diff_mm"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mm": metrics["std_diff_mm"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_present_vs_baseline_with_std.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/prw/MIROC6_prw_mid_SSP585_vs_baseline_with_std.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mm: Change in mean precipitable water (mm)")
print("- Mean_Pct_Change: Percentage change in mean precipitable water (%)")
print("- Std_Diff_mm: Change in standard deviation (mm)")
print("- Std_Pct_Change: Percentage change in standard deviation (%)")


Precipitable Water Analysis (Present - Historical):
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                    (mm)    Change        (mm)    Change
----------------------------------------------------------------------
Global              DJF            2.090      7.92       0.331     69.41
Global              MAM            2.227      8.00       0.495     97.20
Global              JJA            2.489      8.32       0.415     80.71
Global              SON            2.428      8.78       0.416     88.52
Tropics             DJF            3.019      8.24       0.580     62.43
Tropics             MAM            3.193      8.19       0.717     70.56
Tropics             JJA            3.095      7.84       0.484     52.46
Tropics             SON            3.359      8.77       0.590     67.03
Subtropics_N        DJF            1.447      8.82       0.078     20.89
Subtropics_N        MAM            1.942      9.80       0.329     93.07


# Precipitation

In [2]:
# Present Period with Standard Deviation Analysis - PRECIPITATION ANALYSIS (mm/day units)

import xarray as xr
import numpy as np

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Seasonal extraction for PRECIPITATION (using MEAN for mm/day) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average daily precipitation rate
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average daily precipitation rate
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/pr/MIROC6_pr_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/pr/NorESM2-MM_pr_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/pr/MIROC6_pr_future126_20712100_seasonal.nc")

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season (using MEAN for daily rates)
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d = weights.broadcast_like(base_reg["pr"].isel(year=0))

        # CORRECTED: Calculate weighted spatial means for each year
        # Data is now seasonal mean daily precipitation rates (kg m-2 s-1)
        base_spatial_mean_kgm2s = (base_reg["pr"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        present_spatial_mean_kgm2s = (present_reg["pr"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        
        # Convert from kg m-2 s-1 to mm/day
        # 1 kg m-2 s-1 = 86400 mm/day (since 1 kg/m² = 1 mm of water)
        base_spatial_mean = base_spatial_mean_kgm2s * 86400
        present_spatial_mean = present_spatial_mean_kgm2s * 86400
        
        # Calculate period means (average seasonal mean daily rate)
        base_mean = base_spatial_mean.mean(dim="year")
        present_mean = present_spatial_mean.mean(dim="year")
        
        # Calculate standard deviation from the spatially-averaged seasonal mean daily rates
        base_std = base_spatial_mean.std(dim="year", ddof=1)
        present_std = present_spatial_mean.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100
        
        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_mm": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mm": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nPrecipitation Analysis (Present - Historical):")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm/day)':>12}{'Change':>10}{'(mm/day)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mm']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mm']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mm": metrics["mean_diff_mm"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mm": metrics["std_diff_mm"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/pr/NorESM2-MM_pr_present_vs_baseline_with_std.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/pr/MIROC6_pr_long_SSP126_vs_baseline_with_std.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mm: Change in mean daily precipitation (mm/day)")
print("- Mean_Pct_Change: Percentage change in mean daily precipitation (%)")
print("- Std_Diff_mm: Change in standard deviation of seasonal mean daily precipitation (mm/day)")
print("- Std_Pct_Change: Percentage change in standard deviation (%)")
print("\nNote: Results show seasonal mean daily precipitation rates")


Precipitation Analysis (Present - Historical):
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                (mm/day)    Change    (mm/day)    Change
----------------------------------------------------------------------
Global              DJF            0.068      2.00      -0.000     -0.25
Global              MAM            0.071      2.10       0.006     26.71
Global              JJA            0.062      1.78      -0.000     -1.27
Global              SON            0.065      1.96      -0.002    -12.22
Tropics             DJF            0.093      2.26       0.002      2.95
Tropics             MAM            0.076      1.91       0.005     11.33
Tropics             JJA            0.087      2.16      -0.001     -1.53
Tropics             SON            0.073      1.82      -0.002     -4.60
Subtropics_N        DJF           -0.071     -2.97       0.078     44.49
Subtropics_N        MAM            0.118      5.69       0.004      3.63
Subtr

# Evaporation

In [ ]:
# Present Period with Standard Deviation Analysis - EVAPORATION ANALYSIS (mm/day units)

import xarray as xr
import numpy as np

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Seasonal extraction for EVAPORATION (using MEAN for mm/day) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average daily evaporation rate
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average daily evaporation rate
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_future585_20712100_seasonal.nc")

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season (using MEAN for daily rates)
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d = weights.broadcast_like(base_reg["hfls"].isel(year=0))

        # Calculate weighted spatial means for each year
        # Data is latent heat flux in W/m²
        base_spatial_mean_wm2 = (base_reg["hfls"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        present_spatial_mean_wm2 = (present_reg["hfls"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        
        # Convert from W/m² to mm/day for evaporation
        # Latent heat of vaporization = 2.45 × 10^6 J/kg
        # 1 W/m² = 86400 J/(m²·day)
        # 1 kg/m² = 1 mm of water
        # Evaporation (mm/day) = Latent heat flux (W/m²) × 86400 / (2.45 × 10^6)
        base_spatial_mean = base_spatial_mean_wm2 * 86400 / (2.45e6)  # Convert to mm/day
        present_spatial_mean = present_spatial_mean_wm2 * 86400 / (2.45e6)  # Convert to mm/day
        
        # Calculate period means (average seasonal mean daily evaporation rate)
        base_mean = base_spatial_mean.mean(dim="year")
        present_mean = present_spatial_mean.mean(dim="year")
        
        # Calculate standard deviation from the spatially-averaged seasonal mean daily rates
        base_std = base_spatial_mean.std(dim="year", ddof=1)
        present_std = present_spatial_mean.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100
        
        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_mm": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mm": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nEvaporation Analysis (Present - Historical):")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm/day)':>12}{'Change':>10}{'(mm/day)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mm']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mm']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mm": metrics["mean_diff_mm"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mm": metrics["std_diff_mm"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_present_vs_baseline_with_std.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_long_SSP585_vs_baseline_with_std.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mm: Change in mean daily evaporation (mm/day)")
print("- Mean_Pct_Change: Percentage change in mean daily evaporation (%)")
print("- Std_Diff_mm: Change in standard deviation of seasonal mean daily evaporation (mm/day)")
print("- Std_Pct_Change: Percentage change in standard deviation (%)")
print("\nNote: Results show seasonal mean daily evaporation rates converted from latent heat flux (hfls)")


Evaporation Analysis (Present - Historical):
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                (mm/day)    Change    (mm/day)    Change
----------------------------------------------------------------------
Global              DJF            0.133      4.06       0.017     72.00
Global              MAM            0.164      5.09       0.009     23.36
Global              JJA            0.139      4.14       0.003      9.18
Global              SON            0.128      4.02       0.004     17.98
Tropics             DJF            0.165      3.91       0.025     60.94
Tropics             MAM            0.244      6.08       0.020     28.14
Tropics             JJA            0.237      5.59      -0.014    -20.03
Tropics             SON            0.158      3.94       0.002      3.80
Subtropics_N        DJF            0.289      8.37       0.023     23.70
Subtropics_N        MAM            0.271     10.35       0.006      9.97
Subtrop

# Soil Moisture

In [ ]:
# Present Period with Standard Deviation Analysis - SOIL MOISTURE ANALYSIS (m³/m³ units)

import xarray as xr
import numpy as np

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Seasonal extraction for SOIL MOISTURE (using MEAN) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average seasonal soil moisture
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average seasonal soil moisture
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_future126_20712100_seasonal.nc")

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

# --- Constants for conversion ---
soil_depth = 0.1  # meters (10 cm soil layer - standard for mrsos)
water_density = 1000  # kg/m³

for season in seasons:
    # Extract and slice per season
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d = weights.broadcast_like(base_reg["mrsos"].isel(year=0))

        # Calculate weighted spatial means for each year
        # Data is in kg/m² (mass of water per unit area)
        base_spatial_mean_kgm2 = (base_reg["mrsos"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        present_spatial_mean_kgm2 = (present_reg["mrsos"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        
        # Convert from kg/m² to m³/m³ (volumetric water content)
        # kg/m² = mm of water equivalent
        # Volumetric water content = (water depth in m) / (soil depth in m)
        # mrsos (kg/m²) / water_density (kg/m³) = water depth (m)
        # volumetric content = water depth (m) / soil depth (m)
        base_spatial_mean_m3m3 = base_spatial_mean_kgm2 / (water_density * soil_depth)
        present_spatial_mean_m3m3 = present_spatial_mean_kgm2 / (water_density * soil_depth)
        
        # Calculate period means (average seasonal soil moisture)
        base_mean = base_spatial_mean_m3m3.mean(dim="year")
        present_mean = present_spatial_mean_m3m3.mean(dim="year")
        
        # Calculate standard deviation from the spatially-averaged seasonal means
        base_std = base_spatial_mean_m3m3.std(dim="year", ddof=1)
        present_std = present_spatial_mean_m3m3.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100
        
        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_m3m3": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_m3m3": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nSoil Moisture Analysis (Present - Historical):")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(m³/m³)':>12}{'Change':>10}{'(m³/m³)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_m3m3']:>12.5f}{data['mean_pct_change']:>10.2f}{data['std_diff_m3m3']:>12.5f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_m3m3": metrics["mean_diff_m3m3"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_m3m3": metrics["std_diff_m3m3"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_present_vs_baseline_with_std.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_long_SSP126_vs_baseline_with_std.csv", index=False)


print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_m3m3: Change in mean seasonal soil moisture (m³/m³)")
print("- Mean_Pct_Change: Percentage change in mean seasonal soil moisture (%)")
print("- Std_Diff_m3m3: Change in standard deviation of seasonal mean soil moisture (m³/m³)")
print("- Std_Pct_Change: Percentage change in standard deviation (%)")
print(f"\nNote: Soil moisture converted from mrsos (kg/m²) to volumetric water content (m³/m³)")
print(f"Conversion assumes {soil_depth*100} cm soil layer depth")


Soil Moisture Analysis (Present - Historical):
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                 (m³/m³)    Change     (m³/m³)    Change
----------------------------------------------------------------------
Global              DJF         -0.00079     -1.03    -0.00005     -6.50
Global              MAM         -0.00140     -1.85    -0.00010    -11.67
Global              JJA         -0.00173     -2.47    -0.00000     -0.40
Global              SON         -0.00120     -1.68     0.00002      1.85
Tropics             DJF         -0.00022     -0.33    -0.00011     -5.39
Tropics             MAM         -0.00090     -1.34     0.00002      0.77
Tropics             JJA         -0.00071     -1.12    -0.00017     -8.09
Tropics             SON         -0.00026     -0.39     0.00005      2.27
Subtropics_N        DJF         -0.00188     -2.25    -0.00015     -3.89
Subtropics_N        MAM         -0.00189     -2.30     0.00045     18.46
Subtr

# Runoff

In [3]:
# Present Period with Standard Deviation Analysis - RUNOFF ANALYSIS (mm/day units)

import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Seasonal extraction for RUNOFF (using MEAN for mm/day) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average daily runoff rate
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average daily runoff rate
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrro/NorESM2-MM_mrro_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_future585_20712100_seasonal.nc")

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season (using MEAN for daily rates)
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d = weights.broadcast_like(base_reg["mrro"].isel(year=0))

        # Calculate weighted spatial means for each year
        # Data is now seasonal mean daily runoff rates (kg m-2 s-1)
        base_spatial_mean_kgm2s = (base_reg["mrro"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        present_spatial_mean_kgm2s = (present_reg["mrro"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
        
        # Convert from kg m-2 s-1 to mm/day
        # 1 kg m-2 s-1 = 86400 mm/day (since 1 kg/m² = 1 mm of water)
        base_spatial_mean = base_spatial_mean_kgm2s * 86400
        present_spatial_mean = present_spatial_mean_kgm2s * 86400
        
        # Calculate period means (average seasonal mean daily runoff rate)
        base_mean = base_spatial_mean.mean(dim="year")
        present_mean = present_spatial_mean.mean(dim="year")
        
        # Calculate standard deviation from the spatially-averaged seasonal mean daily runoff rates
        base_std = base_spatial_mean.std(dim="year", ddof=1)
        present_std = present_spatial_mean.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100
        
        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        # FIXED: Consistent variable naming
        results[region][season] = {
            "mean_diff_mmday": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mmday": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nRunoff Analysis (Present - Historical):")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm/day)':>12}{'Change':>10}{'(mm/day)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mmday']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mmday']:>12.3f}{data['std_pct_change']:>10.2f}")

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mmday": metrics["mean_diff_mmday"],  # FIXED: Now matches dictionary key
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mmday": metrics["std_diff_mmday"],    # FIXED: Now matches dictionary key
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrro/NorESM2-MM_mrro_present_vs_baseline_with_std.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_long_SSP585_vs_baseline_with_std.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mmday: Change in mean daily runoff (mm/day)")          # FIXED: Matches CSV column name
print("- Mean_Pct_Change: Percentage change in mean daily runoff (%)")
print("- Std_Diff_mmday: Change in standard deviation of seasonal mean daily runoff (mm/day)")  # FIXED: Matches CSV column name
print("- Std_Pct_Change: Percentage change in standard deviation (%)")
print("\nNote: Results show seasonal mean daily runoff rates")


Runoff Analysis (Present - Historical):
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                (mm/day)    Change    (mm/day)    Change
----------------------------------------------------------------------
Global              DJF            0.037     13.40       0.004     18.73
Global              MAM            0.023      7.01       0.006     43.29
Global              JJA            0.024      8.35       0.006     39.41
Global              SON            0.047     17.52       0.012     58.97
Tropics             DJF            0.049     11.87       0.014     33.51
Tropics             MAM            0.049     12.18       0.011     36.87
Tropics             JJA            0.052     15.14       0.007     17.88
Tropics             SON            0.067     15.88       0.027     62.81
Subtropics_N        DJF           -0.002     -3.63       0.001      3.14
Subtropics_N        MAM            0.081     28.39       0.013     24.91
Subtropics_N